# Omega Safe SeqEdit Real-World Benchmark

This notebook is intentionally thin. It calls scripts and loads saved JSON outputs. Core logic lives in `src/omega_safe_seqedit`.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import subprocess
import sys

def find_repo(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'combined_solution' / 'omega_safe_seqedit']
    for path in candidates:
        if (path / 'src' / 'omega_safe_seqedit').exists() and (path / 'scripts').exists():
            return path
    raise RuntimeError('Could not find omega_safe_seqedit repo; start Jupyter from the repo or LRS root.')

REPO = find_repo(Path.cwd())
print(REPO)
sys.path.insert(0, str(REPO / 'src'))


## Choose Preset

Use `mac_debug` first. For real or larger validation runs, switch `PRESET` or use the benchmark-plan cells below.

In [ ]:
PRESET = 'hg002_old_12x_mac'  # Main real-data target; use 'mac_debug' only for smoke tests.
RUN_PREPROCESS = False
RUN_BASELINES = False
RUN_TARGET_ONLY = False
RUN_FULL = False
RUN_EVALUATION = False
CONFIG = REPO / 'configs' / f'{PRESET}.yaml'
CONFIG


In [ ]:
def run_cmd(args):
    print(' '.join(str(a) for a in args))
    return subprocess.run(args, cwd=REPO, check=True)

if RUN_PREPROCESS:
    run_cmd([sys.executable, 'scripts/preprocess_dataset.py', '--config', str(CONFIG)])

## Baselines

Runs no-edit, conservative consensus, support-rule, and any external FASTA/FASTQ baselines listed in the config.

In [ ]:
if RUN_BASELINES:
    run_cmd([sys.executable, 'scripts/run_baselines.py', '--config', str(CONFIG), '--split', 'test'])

## Train SeqEdit Models

`target_only` measures sequence-context priors. `full` adds support/pileup/rule features.

In [ ]:
if RUN_TARGET_ONLY:
    run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(CONFIG), '--run', 'target_only'])
if RUN_FULL:
    run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(CONFIG), '--run', 'full'])

## Evaluate Neural, Rule, and Hybrid Decoding

Hybrid is the safety-first production/debug mode. Neural-only reveals whether the model itself learned the edit decision without rule forcing.

In [ ]:
if RUN_EVALUATION:
    eval_grid = [('target_only', 'neural'), ('full', 'neural'), ('full', 'hybrid')]
    for run_name, mode in eval_grid:
        ckpt = REPO / 'outputs' / PRESET / 'runs' / run_name / 'best.ckpt'
        if ckpt.exists():
            run_cmd([sys.executable, 'scripts/evaluate_predictions.py', '--config', str(CONFIG), '--run', run_name, '--split', 'test', '--mode', mode])

## Compare Summaries

In [ ]:
from omega_safe_seqedit.config import load_config
cfg = load_config(CONFIG)
out = Path(cfg['paths']['output_dir'])
rows = []
baseline_path = out / 'baselines' / 'test_baseline_summary.json'
if baseline_path.exists():
    for name, summary in json.loads(baseline_path.read_text()).items():
        rows.append({'name': name, **{k: summary.get(k) for k in ['usable_score','identity','overcorrection_rate','hard_edit_false_positive_rate','corrected_edits','missed_edits','false_edits']}})
for path in sorted((out / 'runs').glob('*/*_summary.json')) if (out / 'runs').exists() else []:
    summary = json.loads(path.read_text())
    rows.append({'name': '/'.join(path.relative_to(out).parts[:-1]) + '/' + path.stem, **{k: summary.get(k) for k in ['usable_score','identity','overcorrection_rate','hard_edit_false_positive_rate','corrected_edits','missed_edits','false_edits']}})
rows

## False-Edit Audit

The table below is the first thing to inspect before tuning thresholds. It is the safety lens.

In [ ]:
audit_rows = []
for path in sorted((out / 'runs').glob('*/*_summary.json')) if (out / 'runs').exists() else []:
    summary = json.loads(path.read_text())
    for item in summary.get('false_edit_table', [])[:50]:
        audit_rows.append({'source': str(path.relative_to(out)), **item})
audit_rows[:50]

## Qualitative Examples

In [ ]:
pred_path = out / 'runs' / 'full' / 'test_hybrid_predictions.jsonl'
if pred_path.exists():
    examples = [json.loads(line) for line in pred_path.read_text().splitlines() if line.strip()]
    for ex in examples[:3]:
        print('\n---', ex['example_id'], ex.get('case_type'))
        print('target    ', ex['target_seq'][:160])
        print('truth     ', ex['truth_seq'][:160])
        print('prediction', ex['prediction'][:160])
        print('pred_events', ex.get('pred_events', [])[:12])
        print('trace_sample', ex.get('trace', [])[:3])

## Ordered Validation Plan

Synthetic is now a regression lane, not the main objective. By default this plan runs the recommended HG002 old 12x benchmark and then the HG002 audit/calibration cells; turn that switch off if you only want to inspect existing outputs.


In [ ]:

# Synthetic is now a regression lane; HG002 is the optimization target.
RUN_LARGE_NOISY_MULTISEED = False
RUN_SMALL_NOISY_GATE = False
RUN_FALSE_DEL_REGRESSION = False
RUN_LOCAL_REAL_BAM = False
RUN_HG002_CHR20_MAC = False
RUN_HG002_OLD_8X_MAC = False
RUN_HG002_OLD_12X_MAC = True  # Recommended first HG002 safety/usable-score run.
RUN_HG002_OLD_20X_MAC = False

# Analysis/export switches. These run only after the corresponding predictions exist.
RUN_HG002_FALSE_EDIT_AUDIT = True
RUN_BUILD_HG002_ALLOW_GATE_DATASET = True
RUN_CALIBRATE_HG002_ALLOW_GATE = False  # Legacy hard zero-FP threshold; the learned gate below now reports a frontier.
RUN_EXPORT_HG002_SUB_CONTRAST = True
RUN_TRAIN_HG002_SUB_ALLOW_GATE = True
RUN_HG002_COMPARE_DECODE_POLICIES = True

LARGE_NOISY_CONFIG = REPO / 'configs' / 'synthetic_noisy_large_multiseed.yaml'
SMALL_NOISY_CONFIG = REPO / 'configs' / 'synthetic_noisy_small_curated.yaml'
FALSE_DEL_CONFIG = REPO / 'configs' / 'false_del_regression.yaml'
LOCAL_REAL_CONFIG = REPO / 'configs' / 'local_real_mac.yaml'
HG002_CONFIG = REPO / 'configs' / 'hg002_chr20_mac.yaml'
HG002_OLD_8X_CONFIG = REPO / 'configs' / 'hg002_old_8x_mac.yaml'
HG002_OLD_12X_CONFIG = REPO / 'configs' / 'hg002_old_12x_mac.yaml'
HG002_OLD_20X_CONFIG = REPO / 'configs' / 'hg002_old_20x_mac.yaml'

validation_order = [
    ('large_noisy_multiseed_regression', LARGE_NOISY_CONFIG),
    ('small_noisy_gate_regression', SMALL_NOISY_CONFIG),
    ('false_del_regression', FALSE_DEL_CONFIG),
    ('local_real_bam', LOCAL_REAL_CONFIG),
    ('hg002_chr20_mac_raw_bam', HG002_CONFIG),
    ('hg002_old_8x_mac_prewindowed', HG002_OLD_8X_CONFIG),
    ('hg002_old_12x_mac_prewindowed_recommended', HG002_OLD_12X_CONFIG),
    ('hg002_old_20x_mac_prewindowed', HG002_OLD_20X_CONFIG),
]
validation_order


### 1. Large Noisy Synthetic Validation

This uses the seed-47/48 style multiseed benchmark. Acceptance target: `full_hybrid > no_edit` usable, false-edit rate near zero, false DEL count reduced, and neighbor-induced false edits near zero.

In [ ]:
if RUN_LARGE_NOISY_MULTISEED:
    run_cmd([sys.executable, 'scripts/run_multiseed_benchmark.py', '--config', str(LARGE_NOISY_CONFIG)])
    large_summary_path = REPO / 'outputs' / 'synthetic_noisy_large_multiseed' / 'multiseed_summary.json'
    large_summary = json.loads(large_summary_path.read_text())
    large_summary['aggregate']

### 2. Small Noisy Curated Gate

Acceptance target: `corrected_edits >= 14`, `false_edits = 0`, `overcorrection = 0`. This is a fast gate, not the final objective.

In [ ]:
def config_has_real_placeholders(config_path):
    from omega_safe_seqedit.config import load_config
    cfg = load_config(config_path)
    data = cfg.get('data', {})
    if data.get('kind') != 'real_bam':
        return []
    missing = []
    for key in ['bam', 'reference_fasta']:
        value = Path(str(data.get(key, '')))
        if str(value).startswith('/path/to') or not value.exists():
            missing.append(f'{key}: {value}')
    return missing


def run_single_benchmark(config_path, train_target=True, train_full=True):
    missing = config_has_real_placeholders(config_path)
    if missing:
        print('Skipping real-data run because required files are not configured yet:')
        for item in missing:
            print('  -', item)
        print('Edit the YAML config, then rerun this cell.')
        return {'skipped': True, 'missing': missing}
    run_cmd([sys.executable, 'scripts/preprocess_dataset.py', '--config', str(config_path)])
    run_cmd([sys.executable, 'scripts/run_baselines.py', '--config', str(config_path), '--split', 'test'])
    if train_target:
        run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(config_path), '--run', 'target_only'])
        run_cmd([sys.executable, 'scripts/evaluate_predictions.py', '--config', str(config_path), '--run', 'target_only', '--split', 'test', '--mode', 'neural'])
    if train_full:
        run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(config_path), '--run', 'full'])
        run_cmd([sys.executable, 'scripts/evaluate_predictions.py', '--config', str(config_path), '--run', 'full', '--split', 'test', '--mode', 'neural'])
        run_cmd([sys.executable, 'scripts/evaluate_predictions.py', '--config', str(config_path), '--run', 'full', '--split', 'test', '--mode', 'hybrid'])
    run_cmd([sys.executable, 'scripts/export_summary.py', '--config', str(config_path)])
    return {'skipped': False}

if RUN_SMALL_NOISY_GATE:
    run_single_benchmark(SMALL_NOISY_CONFIG)
    small_out = REPO / 'outputs' / 'synthetic_noisy_small_curated'
    json.loads((small_out / 'runs' / 'full' / 'test_hybrid_summary.json').read_text())


### 3. Prior Failure-Mode Regression

Fixed noisy-neighbor/homopolymer false-DEL suite. Acceptance target: false DELs and neighbor-induced false edits stay near zero.

In [ ]:
if RUN_FALSE_DEL_REGRESSION:
    run_single_benchmark(FALSE_DEL_CONFIG, train_target=False, train_full=True)
    reg_out = REPO / 'outputs' / 'false_del_regression'
    json.loads((reg_out / 'runs' / 'full' / 'test_hybrid_summary.json').read_text())

### 4. Local Real BAM Preset

Edit `configs/local_real_mac.yaml` first. Compare: `no_edit`, `consensus`, `support_rule`, `target_only_neural`, `full_neural`, `full_hybrid`, plus external baselines listed in the config. First real goal: `full_hybrid >= consensus` usable with overcorrection near zero.

In [ ]:
if RUN_LOCAL_REAL_BAM:
    result = run_single_benchmark(LOCAL_REAL_CONFIG)
    if not result.get('skipped'):
        local_out = REPO / 'outputs' / 'local_real_mac'
        json.loads((local_out / 'combined_benchmark_summary.json').read_text())
    else:
        result


### 5. HG002 chr20 Mac Preset

Only run this after local real is sane. This should become the first project-report benchmark.

In [ ]:
if RUN_HG002_CHR20_MAC:
    result = run_single_benchmark(HG002_CONFIG)
    if not result.get('skipped'):
        hg002_out = REPO / 'outputs' / 'hg002_chr20_mac'
        json.loads((hg002_out / 'combined_benchmark_summary.json').read_text())
    else:
        result



### 6. Old HG002 chr20 Pre-Windowed Benchmarks

These are now the main real-data target. Synthetic runs are still useful regressions, but the decision criterion is HG002 usable score with near-zero false edits. The recommended first run is 12x because it is substantial enough to expose real false-edit behavior while still being Mac-friendly.

The old HG002 windows include variant/preserve/repeat masks that are imported into the combined schema, so the hybrid decoder and audits can report variant-aware and haplotype-preservation failure modes.


In [ ]:
if RUN_HG002_OLD_8X_MAC:
    run_single_benchmark(HG002_OLD_8X_CONFIG)
    out8 = REPO / 'outputs' / 'hg002_old_8x_mac'
    json.loads((out8 / 'combined_benchmark_summary.json').read_text())


In [ ]:
if RUN_HG002_OLD_12X_MAC:
    run_single_benchmark(HG002_OLD_12X_CONFIG)
    out12 = REPO / 'outputs' / 'hg002_old_12x_mac'
    json.loads((out12 / 'combined_benchmark_summary.json').read_text())


In [ ]:
if RUN_HG002_OLD_20X_MAC:
    run_single_benchmark(HG002_OLD_20X_CONFIG)
    out20 = REPO / 'outputs' / 'hg002_old_20x_mac'
    json.loads((out20 / 'combined_benchmark_summary.json').read_text())


## HG002 Coverage Comparison

After running any of the old HG002 presets, this summarizes no-edit, consensus, support-rule, target-only, full-neural, and full-hybrid outputs across coverage levels.


In [ ]:
def collect_hg002_old_rows():
    rows = []
    for name in ['hg002_old_8x_mac', 'hg002_old_12x_mac', 'hg002_old_20x_mac']:
        root = REPO / 'outputs' / name
        baseline_path = root / 'baselines' / 'test_baseline_summary.json'
        if baseline_path.exists():
            for run_name, summary in json.loads(baseline_path.read_text()).items():
                rows.append({
                    'preset': name,
                    'run': run_name,
                    'usable_score': summary.get('usable_score'),
                    'identity': summary.get('identity'),
                    'overcorrection_rate': summary.get('overcorrection_rate'),
                    'hard_edit_false_positive_rate': summary.get('hard_edit_false_positive_rate'),
                    'corrected_edits': summary.get('total_corrected_edits', summary.get('corrected_edits')),
                    'false_edits': summary.get('total_false_edits', summary.get('false_edits')),
                })
        for summary_path in sorted((root / 'runs').glob('*/*_summary.json')) if (root / 'runs').exists() else []:
            summary = json.loads(summary_path.read_text())
            rows.append({
                'preset': name,
                'run': '/'.join(summary_path.relative_to(root).parts[:-1]) + '/' + summary_path.stem,
                'usable_score': summary.get('usable_score'),
                'identity': summary.get('identity'),
                'overcorrection_rate': summary.get('overcorrection_rate'),
                'hard_edit_false_positive_rate': summary.get('hard_edit_false_positive_rate'),
                'corrected_edits': summary.get('total_corrected_edits', summary.get('corrected_edits')),
                'false_edits': summary.get('total_false_edits', summary.get('false_edits')),
            })
    return rows

collect_hg002_old_rows()



## HG002 False-Edit Safety Audit

Run this after an HG002 old-window benchmark. The current optimization target is **HG002 false SUB reduction**, not synthetic recall. This audit highlights false substitutions in tandem repeats, neighboring-edit ambiguity, variant-rich/protect regions, and low-confidence regions before any threshold tuning.


In [ ]:
HG002_AUDIT_PRESET = 'hg002_old_12x_mac'
HG002_AUDIT_ROOT = REPO / 'outputs' / HG002_AUDIT_PRESET
HG002_RUN_DIR = HG002_AUDIT_ROOT / 'runs' / 'full'
HG002_CALIBRATION_DIR = HG002_AUDIT_ROOT / 'calibration'

# One explicit file per policy. No silent fallback to older tags: if a file is missing,
# the notebook reports it as missing so the final SOTA/readiness cells cannot go stale.
HG002_POLICY_FILES = {
    'full_hybrid_default': {
        'summary': HG002_RUN_DIR / 'test_hybrid_summary.json',
        'predictions': HG002_RUN_DIR / 'test_hybrid_predictions.jsonl',
    },
    'full_hybrid_new': {
        'summary': HG002_RUN_DIR / 'test_hybrid_new_summary.json',
        'predictions': HG002_RUN_DIR / 'test_hybrid_new_predictions.jsonl',
    },
    'strict_no_recovery': {
        'summary': HG002_RUN_DIR / 'test_strict_no_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_strict_no_recovery_predictions.jsonl',
    },
    'sub_recovery': {
        'summary': HG002_RUN_DIR / 'test_sub_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_sub_recovery_predictions.jsonl',
    },
    'ins_recovery': {
        'summary': HG002_RUN_DIR / 'test_ins_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_ins_recovery_predictions.jsonl',
    },
    'del_recovery': {
        'summary': HG002_RUN_DIR / 'test_del_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_del_recovery_predictions.jsonl',
    },
    'combined_recovery': {
        'summary': HG002_RUN_DIR / 'test_combined_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_combined_recovery_predictions.jsonl',
    },
    'ultra_safe_sub_recovery': {
        'summary': HG002_RUN_DIR / 'test_ultra_safe_sub_recovery_summary.json',
        'predictions': HG002_RUN_DIR / 'test_ultra_safe_sub_recovery_predictions.jsonl',
    },
    **{
        f'ranked_sub_top_{k}': {
            'summary': HG002_RUN_DIR / f'test_ranked_sub_top_{k}_summary.json',
            'predictions': HG002_RUN_DIR / f'test_ranked_sub_top_{k}_predictions.jsonl',
        }
        for k in [1, 2, 5, 10, 20]
    },
}
HG002_AUDIT_POLICY = 'full_hybrid_default'

def hg002_policy_file(policy, kind='summary'):
    return HG002_POLICY_FILES[policy][kind]

HG002_HYBRID_SUMMARY = hg002_policy_file(HG002_AUDIT_POLICY, 'summary')
HG002_HYBRID_PREDS = hg002_policy_file(HG002_AUDIT_POLICY, 'predictions')
HG002_CANDIDATES = HG002_CALIBRATION_DIR / 'allow_gate_candidates.jsonl'
HG002_CANDIDATE_SUMMARY = HG002_CALIBRATION_DIR / 'allow_gate_candidate_summary.json'
HG002_ALLOW_THRESHOLDS = HG002_CALIBRATION_DIR / 'allow_gate_thresholds.json'
HG002_LEARNED_ALLOW_GATE = HG002_CALIBRATION_DIR / 'learned_allow_gate.json'
HG002_SUB_CANDIDATES = HG002_CALIBRATION_DIR / 'support_rule_sub_candidates.jsonl'
HG002_SUB_CANDIDATE_SUMMARY = HG002_CALIBRATION_DIR / 'support_rule_sub_candidate_summary.json'
HG002_SUB_RECOVERY_CANDIDATES = HG002_CALIBRATION_DIR / 'sub_recovery_allow_gate_candidates.jsonl'
HG002_SUB_RECOVERY_CANDIDATE_SUMMARY = HG002_CALIBRATION_DIR / 'sub_recovery_allow_gate_candidate_summary.json'
HG002_SUB_RECOVERY_SUB_CANDIDATES = HG002_CALIBRATION_DIR / 'sub_recovery_support_rule_sub_candidates.jsonl'
HG002_SUB_RECOVERY_SUB_CANDIDATE_SUMMARY = HG002_CALIBRATION_DIR / 'sub_recovery_support_rule_sub_candidate_summary.json'
HG002_SUB_LOCAL_CONTEXT = HG002_CALIBRATION_DIR / 'paired_sub_local_context.jsonl'
HG002_SUB_LOCAL_CONTEXT_SUMMARY = HG002_CALIBRATION_DIR / 'paired_sub_local_context_summary.json'
HG002_SUB_RECOVERY_LOCAL_CONTEXT = HG002_CALIBRATION_DIR / 'sub_recovery_paired_sub_local_context.jsonl'
HG002_SUB_RECOVERY_LOCAL_CONTEXT_SUMMARY = HG002_CALIBRATION_DIR / 'sub_recovery_paired_sub_local_context_summary.json'
HG002_SUB_ALLOW_GATE = HG002_CALIBRATION_DIR / 'sub_allow_gate.json'
HG002_RANKED_SUB_DIR = HG002_CALIBRATION_DIR / 'ranked_sub_recovery'
HG002_RANKED_SUB_SUMMARY = HG002_RANKED_SUB_DIR / 'ranked_sub_recovery_summary.json'
HG002_RANKED_SUB_CANDIDATES = HG002_RANKED_SUB_DIR / 'ranked_sub_candidates.jsonl'
HG002_RANK12_CONTEXT = HG002_RANKED_SUB_DIR / 'rank_1_vs_2_local_context.jsonl'
HG002_RANK12_CONTEXT_SUMMARY = HG002_RANKED_SUB_DIR / 'rank_1_vs_2_local_context_summary.json'
HG002_RANKED_SUB_TOP_KS = [1, 2, 5, 10, 20]

REQUESTED_FALSE_EDIT_COLUMNS = [
    'candidate_id', 'example_id', 'position', 'gold_label', 'predicted_label', 'support_rule_label', 'neural_only_label',
    'edit_type', 'target_base', 'truth_base', 'support_base_counts', 'support_ins_base_counts',
    'support_del_count', 'support_ins_count', 'normalized_support_ins_base_counts', 'top_inserted_base_count', 'top_inserted_base_fraction', 'inserted_base_margin', 'raw_inserted_base_total', 'support_depth', 'support_fraction', 'support_margin',
    'entropy', 'consensus_agreement', 'homopolymer_flag', 'homopolymer_run_length', 'tandem_repeat_flag',
    'repeat_flag', 'neighbor_edit_distance', 'boundary_flag', 'variant_mask', 'phased_variant_mask',
    'truth_vcf_overlap', 'preserve_mask', 'uncertainty_label', 'variant_rich_flag', 'confident_bed_status',
    'local_rule_density', 'local_mismatch_density', 'nearby_indel_density', 'left_support_match_fraction', 'right_support_match_fraction',
    'support_forward_fraction', 'support_forward_count', 'support_reverse_count', 'support_strand_bias', 'support_same_haplotype_fraction', 'support_match_fraction',
    'repeat_strength', 'mapping_quality_available', 'mapping_quality_mean', 'reference_kmer_uniqueness_available', 'reference_kmer_uniqueness',
    'main_probs', 'sub_probs', 'ins_probs', 'sub_candidate_details', 'indel_candidate_details', 'accepted_sub_candidate', 'accepted_indel_candidate', 'forced_by_rule', 'vetoed', 'veto_or_rescue_reason',
]

def load_hg002_false_edit_audit(policy=HG002_AUDIT_POLICY):
    summary_path = hg002_policy_file(policy, 'summary')
    if not summary_path.exists():
        return {'policy': policy, 'missing': str(summary_path), 'hint': 'Run the HG002 benchmark/policy comparison cell first.'}
    summary = json.loads(summary_path.read_text())
    rows = summary.get('false_edit_table', [])
    compact_rows = [{key: row.get(key) for key in REQUESTED_FALSE_EDIT_COLUMNS} for row in rows]
    by_type = Counter(row.get('edit_type') for row in compact_rows)
    mechanism = summary.get('false_edit_mechanism_hypotheses', {})
    context = summary.get('false_edit_context_counts', {})
    return {
        'preset': HG002_AUDIT_PRESET,
        'policy': policy,
        'source_summary': str(summary_path),
        'usable_score': summary.get('usable_score'),
        'identity': summary.get('identity'),
        'false_edits_total': summary.get('total_false_edits', summary.get('false_edits')),
        'false_edits_by_type': dict(by_type),
        'false_edit_context_counts': context,
        'false_edit_mechanism_hypotheses': mechanism,
        'top_false_edit_rows': compact_rows[:50],
    }

hg002_false_edit_audit = load_hg002_false_edit_audit() if RUN_HG002_FALSE_EDIT_AUDIT else {'skipped': True}
hg002_false_edit_audit



### HG002 False-SUB Focus Table

This is the first table to inspect. It isolates false substitutions and summarizes how many occur in tandem/repeat, neighboring candidate, variant/protect, or low-confidence contexts.


In [ ]:

def hg002_false_sub_report():
    if not HG002_HYBRID_SUMMARY.exists():
        return {'missing': str(HG002_HYBRID_SUMMARY)}
    summary = json.loads(HG002_HYBRID_SUMMARY.read_text())
    rows = [row for row in summary.get('false_edit_table', []) if row.get('edit_type') == 'SUB']
    return {
        'false_sub_total': len(rows),
        'false_sub_in_tandem_or_repeat': sum(1 for row in rows if row.get('tandem_repeat_flag') or row.get('repeat_flag')),
        'false_sub_neighbor_ambiguous': sum(1 for row in rows if row.get('neighbor_rule_flag')),
        'false_sub_variant_or_preserve': sum(1 for row in rows if row.get('truth_vcf_overlap') or row.get('preserve_mask') or row.get('variant_rich_flag')),
        'false_sub_low_confidence': sum(1 for row in rows if row.get('low_confidence_or_preserve')),
        'false_sub_examples': [{key: row.get(key) for key in REQUESTED_FALSE_EDIT_COLUMNS} for row in rows[:50]],
    }

hg002_false_sub_report() if RUN_HG002_FALSE_EDIT_AUDIT else {'skipped': True}



### HG002 Would-Have-Corrected True-Edit Audit

Hybrid may now veto most indels for safety. This table shows true hard edits that support-rule would have corrected but hybrid left as COPY, including veto reason, repeat/tandem/neighbor flags, support evidence, and neural probabilities. Use this only after false edits are near zero to recover a tiny safe subset.


In [ ]:
VETOED_TRUE_EDIT_COLUMNS = [
    'example_id', 'position', 'gold_label', 'support_rule_label', 'neural_label', 'hybrid_label',
    'veto_reason', 'edit_type', 'support_depth', 'support_fraction', 'support_margin', 'entropy',
    'repeat_flag', 'tandem_repeat_flag', 'homopolymer_run_length', 'neighbor_rule_flag', 'boundary_flag',
    'confident_bed_status', 'variant_mask', 'truth_vcf_overlap', 'neural_main_probs', 'payload_probs',
    'candidate_allow_score', 'safe_recovery_score', 'sub_candidate_details', 'indel_candidate_details',
    'normalized_support_ins_base_counts', 'top_inserted_base_fraction', 'inserted_base_margin',
    'local_rule_density', 'local_mismatch_density', 'nearby_indel_density', 'left_support_match_fraction',
    'right_support_match_fraction', 'support_forward_fraction', 'support_forward_count', 'support_reverse_count', 'support_strand_bias',
    'support_same_haplotype_fraction', 'support_match_fraction', 'repeat_strength',
]

def hg002_vetoed_true_edit_audit(limit=50):
    if not HG002_HYBRID_SUMMARY.exists():
        return {'missing': str(HG002_HYBRID_SUMMARY)}
    summary = json.loads(HG002_HYBRID_SUMMARY.read_text())
    rows = summary.get('vetoed_true_support_rule_table', summary.get('would_have_corrected_table', summary.get('support_rule_gap_table', [])))
    rows = sorted(rows, key=lambda row: row.get('safe_recovery_score', -999), reverse=True)
    by_type = Counter(row.get('edit_type', row.get('gold_type')) for row in rows)
    safest = [{key: row.get(key) for key in VETOED_TRUE_EDIT_COLUMNS} for row in rows[:limit]]
    false_subs = [row for row in summary.get('false_edit_table', []) if row.get('edit_type') == 'SUB']
    false_subs = sorted(false_subs, key=lambda row: row.get('safe_recovery_score', -999), reverse=True)
    return {
        'policy': HG002_AUDIT_POLICY,
        'source_summary': str(HG002_HYBRID_SUMMARY),
        'vetoed_true_edit_total': len(rows),
        'by_edit_type': dict(by_type),
        'top_50_safest_true_edits': safest,
        'top_false_sub_examples_for_contrast': [{key: row.get(key) for key in VETOED_TRUE_EDIT_COLUMNS} for row in false_subs[:50]],
        'safest_candidate_count_SUB': sum(1 for row in rows[:limit] if row.get('edit_type', row.get('gold_type')) == 'SUB'),
        'repeat_or_tandem': sum(1 for row in rows if row.get('repeat_flag') or row.get('tandem_repeat_flag')),
        'neighbor_ambiguous': sum(1 for row in rows if row.get('neighbor_rule_flag')),
    }

hg002_vetoed_true_edit_audit() if RUN_HG002_FALSE_EDIT_AUDIT else {'skipped': True}



### Insertion Feature Normalization Check

This verifies that current audit outputs use normalized insertion payload fractions. If any rows report `top_inserted_base_fraction > 1`, rerun evaluation after the normalization patch before trusting insertion recovery.


In [ ]:

def hg002_insertion_fraction_sanity_check():
    if not HG002_HYBRID_SUMMARY.exists():
        return {'missing': str(HG002_HYBRID_SUMMARY)}
    summary = json.loads(HG002_HYBRID_SUMMARY.read_text())
    tables = []
    for key in ['false_edit_table', 'vetoed_true_support_rule_table', 'would_have_corrected_table', 'support_rule_gap_table']:
        tables.extend({'source_table': key, **row} for row in summary.get(key, []))
    bad_rows = [row for row in tables if (row.get('top_inserted_base_fraction') is not None and row.get('top_inserted_base_fraction') > 1.000001)]
    return {
        'checked_rows': len(tables),
        'bad_fraction_rows': len(bad_rows),
        'max_top_inserted_base_fraction': max([row.get('top_inserted_base_fraction', 0) or 0 for row in tables], default=0),
        'examples': bad_rows[:20],
    }

hg002_insertion_fraction_sanity_check() if RUN_HG002_FALSE_EDIT_AUDIT else {'skipped': True}



## HG002 Real-Data Allow/Edit Gate Calibration

These cells prepare a real-data calibration dataset from decoded HG002 candidate edits. The target is binary: candidate edit is safe to apply or should abstain. Use this after the false-edit audit; it is deliberately separate from model training so you can tune safety without hiding the mechanism.


In [ ]:

if RUN_BUILD_HG002_ALLOW_GATE_DATASET:
    if HG002_HYBRID_PREDS.exists():
        run_cmd([
            sys.executable,
            'scripts/build_allow_gate_dataset.py',
            '--predictions', str(HG002_HYBRID_PREDS),
            '--output', str(HG002_CANDIDATES),
            '--summary-output', str(HG002_CANDIDATE_SUMMARY),
        ])
        json.loads(HG002_CANDIDATE_SUMMARY.read_text())
    else:
        {'missing': str(HG002_HYBRID_PREDS), 'hint': 'Run full_hybrid evaluation first.'}
else:
    {'skipped': True}


### False-SUB vs True-SUB Candidate Contrast

This exports scalar and local-context inspection tables for HG002 support-rule SUB candidates. The paired local-context JSONL is the important one when scalar features fail: it includes ±20 bp target/truth/reference context, support pileup, support reads contributing to the majority, nearby VCF records when available, and repeat/low-complexity annotations.


In [ ]:
def export_sub_contrast_from_candidates(candidate_path, output_path, summary_path):
    run_cmd([
        sys.executable,
        'scripts/export_sub_candidate_contrast.py',
        '--candidates', str(candidate_path),
        '--output', str(output_path),
        '--summary-output', str(summary_path),
    ])
    return json.loads(summary_path.read_text())

def export_sub_local_context(candidate_path, predictions_path, output_path, summary_path):
    run_cmd([
        sys.executable,
        'scripts/export_sub_local_context.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--candidates', str(candidate_path),
        '--predictions', str(predictions_path),
        '--output', str(output_path),
        '--summary-output', str(summary_path),
        '--limit-per-class', '50',
        '--flank', '20',
        '--score-radius', '5',
    ])
    return json.loads(summary_path.read_text())

if RUN_EXPORT_HG002_SUB_CONTRAST:
    reports = {}
    if HG002_CANDIDATES.exists() and HG002_HYBRID_PREDS.exists():
        reports['current_policy_candidates'] = export_sub_contrast_from_candidates(
            HG002_CANDIDATES,
            HG002_SUB_CANDIDATES,
            HG002_SUB_CANDIDATE_SUMMARY,
        )
        reports['current_policy_local_context'] = export_sub_local_context(
            HG002_CANDIDATES,
            HG002_HYBRID_PREDS,
            HG002_SUB_LOCAL_CONTEXT,
            HG002_SUB_LOCAL_CONTEXT_SUMMARY,
        )
    else:
        reports['current_policy_candidates'] = {'missing': [str(HG002_CANDIDATES), str(HG002_HYBRID_PREDS)], 'hint': 'Build the candidate table and full_hybrid predictions first.'}

    # If the previous/explicit SUB-recovery branch exists, build a separate contrast table from it.
    # This is the table to inspect for true recovered SUBs vs false recovered SUBs.
    sub_recovery_preds = hg002_policy_file('sub_recovery', 'predictions')
    if sub_recovery_preds.exists():
        run_cmd([
            sys.executable,
            'scripts/build_allow_gate_dataset.py',
            '--predictions', str(sub_recovery_preds),
            '--output', str(HG002_SUB_RECOVERY_CANDIDATES),
            '--summary-output', str(HG002_SUB_RECOVERY_CANDIDATE_SUMMARY),
        ])
        reports['sub_recovery_candidates'] = json.loads(HG002_SUB_RECOVERY_CANDIDATE_SUMMARY.read_text())
        reports['sub_recovery_true_vs_false_sub'] = export_sub_contrast_from_candidates(
            HG002_SUB_RECOVERY_CANDIDATES,
            HG002_SUB_RECOVERY_SUB_CANDIDATES,
            HG002_SUB_RECOVERY_SUB_CANDIDATE_SUMMARY,
        )
        reports['sub_recovery_local_context'] = export_sub_local_context(
            HG002_SUB_RECOVERY_CANDIDATES,
            sub_recovery_preds,
            HG002_SUB_RECOVERY_LOCAL_CONTEXT,
            HG002_SUB_RECOVERY_LOCAL_CONTEXT_SUMMARY,
        )
    else:
        reports['sub_recovery_candidates'] = {'missing': str(sub_recovery_preds), 'hint': 'Run the A/B sub_recovery policy cell, then rerun this contrast cell.'}
    reports
else:
    {'skipped': True}


In [ ]:
# Legacy threshold calibration is intentionally off by default now. The hard zero-FP rule often selected
# threshold=1.01, meaning “allow nothing.” Use the learned allow-gate cell below instead; it prints a
# precision/recall frontier so we can look for thresholds with 1-5 true edits and 0 false edits, or
# 20+ true edits with <=1 false edit.
if RUN_CALIBRATE_HG002_ALLOW_GATE:
    if HG002_CANDIDATES.exists():
        run_cmd([
            sys.executable,
            'scripts/calibrate_allow_gate.py',
            '--candidates', str(HG002_CANDIDATES),
            '--output', str(HG002_ALLOW_THRESHOLDS),
            '--max-false-positives', '0',
        ])
        json.loads(HG002_ALLOW_THRESHOLDS.read_text())
    else:
        {'missing': str(HG002_CANDIDATES), 'hint': 'Build the candidate table first.'}
else:
    {'skipped': True, 'replacement': 'scripts/train_allow_gate.py frontier output in the next cell'}


### SUB-Specific Learned Allow Gate

The hand-tuned SUB recovery rule is disabled by default because it produced false edits. This trains an edit-type-specific candidate classifier and should be used only as an explicit A/B recovery experiment optimized for zero false positives first.


In [ ]:
if RUN_TRAIN_HG002_SUB_ALLOW_GATE:
    if HG002_CANDIDATES.exists():
        run_cmd([
            sys.executable,
            'scripts/train_allow_gate.py',
            '--candidates', str(HG002_CANDIDATES),
            '--output', str(HG002_SUB_ALLOW_GATE),
            '--max-false-positives', '0',
            '--max-false-positive-rate', '0.0',
            '--candidate-source', 'support_rule',
            '--edit-types', 'SUB',
        ])
        gate = json.loads(HG002_SUB_ALLOW_GATE.read_text())
        sub_model = gate.get('models', {}).get('SUB', {})
        {
            'output': str(HG002_SUB_ALLOW_GATE),
            'selected_threshold': sub_model.get('threshold'),
            'selected_validation': sub_model.get('validation'),
            'frontier_metric_note': sub_model.get('frontier_metric_note'),
            'zero_fp_1_to_5_true_edit_thresholds': sub_model.get('zero_fp_1_to_5_true_edit_thresholds', [])[:10],
            'twenty_true_edits_le_one_false_positive_thresholds': sub_model.get('twenty_true_edits_le_one_false_positive_thresholds', [])[:10],
            'frontier_preview': sub_model.get('frontier', [])[:25],
            'pareto_operating_points': sub_model.get('pareto_operating_points', []),
        }
    else:
        {'missing': str(HG002_CANDIDATES), 'hint': 'Build the HG002 candidate table first.'}
else:
    {'skipped': True}


def build_ranked_sub_recovery_reports():
    if not HG002_CANDIDATES.exists():
        return {'missing': str(HG002_CANDIDATES), 'hint': 'Build the HG002 candidate table first.'}
    run_cmd([
        sys.executable,
        'scripts/rank_sub_recovery_candidates.py',
        '--candidates', str(HG002_CANDIDATES),
        '--output-dir', str(HG002_RANKED_SUB_DIR),
        '--summary-output', str(HG002_RANKED_SUB_SUMMARY),
        '--top-k', ','.join(str(k) for k in HG002_RANKED_SUB_TOP_KS),
        '--min-local-gain', '0.25',
        '--ranking-mode', 'pairwise',
    ])
    return json.loads(HG002_RANKED_SUB_SUMMARY.read_text())

ranked_sub_recovery_report = build_ranked_sub_recovery_reports() if RUN_TRAIN_HG002_SUB_ALLOW_GATE else {'skipped': True}
ranked_sub_recovery_report


def export_rank_1_vs_rank_2_context():
    if not HG002_RANKED_SUB_CANDIDATES.exists() or not HG002_HYBRID_PREDS.exists():
        return {'missing': [str(HG002_RANKED_SUB_CANDIDATES), str(HG002_HYBRID_PREDS)], 'hint': 'Build ranked SUB recovery reports first.'}
    run_cmd([
        sys.executable,
        'scripts/export_sub_local_context.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--candidates', str(HG002_CANDIDATES),
        '--predictions', str(HG002_HYBRID_PREDS),
        '--ranked-candidates', str(HG002_RANKED_SUB_CANDIDATES),
        '--ranks', '1,2',
        '--output', str(HG002_RANK12_CONTEXT),
        '--summary-output', str(HG002_RANK12_CONTEXT_SUMMARY),
        '--flank', '30',
        '--score-radius', '10',
    ])
    return {
        'summary': json.loads(HG002_RANK12_CONTEXT_SUMMARY.read_text()),
        'rows': read_jsonl(HG002_RANK12_CONTEXT),
        'question': 'What distinguishes rank 1 true positive from rank 2 false positive?',
    }

rank_1_vs_rank_2_context = export_rank_1_vs_rank_2_context() if RUN_TRAIN_HG002_SUB_ALLOW_GATE else {'skipped': True}
rank_1_vs_rank_2_context


### A/B Safe Recovery Report

This always separates recovery components: strict no-recovery, SUB recovery, INS recovery, DEL recovery, and combined recovery. SUB recovery is off by default and must be explicitly enabled here for an A/B run.


In [ ]:
if RUN_HG002_COMPARE_DECODE_POLICIES:
    # Current explicit hybrid policy output. This is intentionally distinct from the default benchmark file.
    run_cmd([
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--disable-safe-recovery',
        '--output-tag', 'hybrid_new',
    ])
    # Strict safety baseline: no recovery. This should preserve false_edits=0.
    run_cmd([
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--disable-safe-recovery',
        '--output-tag', 'strict_no_recovery',
    ])

    run_cmd([
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--ultra-safe-sub-recovery',
        '--output-tag', 'ultra_safe_sub_recovery',
    ])

    if HG002_RANKED_SUB_SUMMARY.exists():
        ranked_summary = json.loads(HG002_RANKED_SUB_SUMMARY.read_text())
        allowlists = {item['top_k']: item['allowlist'] for item in ranked_summary.get('top_k_reports', [])}
        for k in HG002_RANKED_SUB_TOP_KS:
            allowlist_path = allowlists.get(k)
            if allowlist_path:
                run_cmd([
                    sys.executable, 'scripts/evaluate_predictions.py',
                    '--config', str(HG002_OLD_12X_CONFIG),
                    '--run', 'full', '--split', 'test', '--mode', 'hybrid',
                    '--ranked-sub-recovery-allowlist', str(allowlist_path),
                    '--output-tag', f'ranked_sub_top_{k}',
                ])
    # SUB-only recovery: explicit A/B branch, disabled in config defaults.
    sub_cmd = [
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--enable-safe-recovery', '--safe-recovery-edit-types', 'SUB',
        '--output-tag', 'sub_recovery',
    ]
    if HG002_SUB_ALLOW_GATE.exists():
        sub_cmd.extend(['--allow-gate', str(HG002_SUB_ALLOW_GATE)])
    run_cmd(sub_cmd)
    # INS/DEL recovery are reported as explicit branches but remain intentionally strict unless implemented later.
    run_cmd([
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--enable-safe-recovery', '--safe-recovery-edit-types', 'INS',
        '--output-tag', 'ins_recovery',
    ])
    run_cmd([
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--enable-safe-recovery', '--safe-recovery-edit-types', 'DEL',
        '--output-tag', 'del_recovery',
    ])
    combined_cmd = [
        sys.executable, 'scripts/evaluate_predictions.py',
        '--config', str(HG002_OLD_12X_CONFIG),
        '--run', 'full', '--split', 'test', '--mode', 'hybrid',
        '--enable-safe-recovery', '--safe-recovery-edit-types', 'SUB,INS,DEL',
        '--output-tag', 'combined_recovery',
    ]
    if HG002_SUB_ALLOW_GATE.exists():
        combined_cmd.extend(['--allow-gate', str(HG002_SUB_ALLOW_GATE)])
    run_cmd(combined_cmd)
else:
    {'skipped': True}


In [ ]:
def hg002_full_comparison_table(preset_name='hg002_old_12x_mac'):
    root = REPO / 'outputs' / preset_name
    rows = []
    baseline_path = root / 'baselines' / 'test_baseline_summary.json'
    if baseline_path.exists():
        for name, summary in json.loads(baseline_path.read_text()).items():
            rows.append({'run': name, 'source_file': str(baseline_path), **{k: summary.get(k) for k in ['usable_score', 'hard_edit_false_positive_rate', 'total_false_edits', 'total_corrected_edits', 'identity', 'overcorrection_rate']}})
    policy_files = HG002_POLICY_FILES if preset_name == HG002_AUDIT_PRESET else {
        'full_hybrid_default': {'summary': root / 'runs' / 'full' / 'test_hybrid_summary.json'},
        'full_hybrid_new': {'summary': root / 'runs' / 'full' / 'test_hybrid_new_summary.json'},
        'strict_no_recovery': {'summary': root / 'runs' / 'full' / 'test_strict_no_recovery_summary.json'},
        'sub_recovery': {'summary': root / 'runs' / 'full' / 'test_sub_recovery_summary.json'},
        'ins_recovery': {'summary': root / 'runs' / 'full' / 'test_ins_recovery_summary.json'},
        'del_recovery': {'summary': root / 'runs' / 'full' / 'test_del_recovery_summary.json'},
        'combined_recovery': {'summary': root / 'runs' / 'full' / 'test_combined_recovery_summary.json'},
        'ultra_safe_sub_recovery': {'summary': root / 'runs' / 'full' / 'test_ultra_safe_sub_recovery_summary.json'},
        **{f'ranked_sub_top_{k}': {'summary': root / 'runs' / 'full' / f'test_ranked_sub_top_{k}_summary.json'} for k in [1, 2, 5, 10, 20]},
    }
    neural_path = root / 'runs' / 'full' / 'test_neural_summary.json'
    if neural_path.exists():
        summary = json.loads(neural_path.read_text())
        rows.append({'run': 'full_neural', 'source_file': str(neural_path), **{k: summary.get(k) for k in ['usable_score', 'hard_edit_false_positive_rate', 'total_false_edits', 'total_corrected_edits', 'identity', 'overcorrection_rate']}})
    for label in ['full_hybrid_default', 'full_hybrid_new', 'strict_no_recovery', 'sub_recovery', 'ins_recovery', 'del_recovery', 'combined_recovery', 'ultra_safe_sub_recovery'] + [f'ranked_sub_top_{k}' for k in [1, 2, 5, 10, 20]]:
        path = policy_files[label]['summary']
        if path.exists():
            summary = json.loads(path.read_text())
            rows.append({'run': label, 'source_file': str(path), **{k: summary.get(k) for k in ['usable_score', 'hard_edit_false_positive_rate', 'total_false_edits', 'total_corrected_edits', 'identity', 'overcorrection_rate']}})
        else:
            rows.append({'run': label, 'missing': str(path)})
    return rows

hg002_full_comparison_table(HG002_AUDIT_PRESET)



## SOTA-Readiness Gate

Do not compare to external SOTA tools until the internal real-data gates are passed: `full_hybrid` should beat `no_edit`, conservative consensus, and support-rule on usable score, with false edits near zero. This cell reports whether the current HG002 run is ready for external comparison.


In [ ]:
def sota_readiness_report(preset_name='hg002_old_12x_mac'):
    root = REPO / 'outputs' / preset_name
    baseline_path = root / 'baselines' / 'test_baseline_summary.json'
    policy_rows = hg002_full_comparison_table(preset_name)
    if not baseline_path.exists():
        return {'ready_for_external_sota_claims': False, 'missing': [str(baseline_path)]}
    baselines = json.loads(baseline_path.read_text())
    required_baselines = ['no_edit', 'consensus', 'support_rule']
    required_policies = ['full_hybrid_new', 'full_hybrid_default', 'strict_no_recovery', 'sub_recovery', 'ins_recovery', 'del_recovery', 'combined_recovery', 'ultra_safe_sub_recovery'] + [f'ranked_sub_top_{k}' for k in [1, 2, 5, 10, 20]]
    policy_by_name = {row['run']: row for row in policy_rows if row.get('run') in required_policies}
    policy_reports = []
    for policy in required_policies:
        row = policy_by_name.get(policy, {'run': policy, 'missing': 'not reported'})
        comparisons = {}
        for baseline in required_baselines:
            baseline_usable = baselines.get(baseline, {}).get('usable_score')
            policy_usable = row.get('usable_score')
            comparisons[baseline] = {
                'baseline_usable': baseline_usable,
                'policy_usable': policy_usable,
                'beats': policy_usable is not None and baseline_usable is not None and policy_usable > baseline_usable,
            }
        false_edits = row.get('total_false_edits')
        policy_reports.append({
            'policy': policy,
            'source_file': row.get('source_file'),
            'missing': row.get('missing'),
            'usable_score': row.get('usable_score'),
            'identity': row.get('identity'),
            'total_false_edits': false_edits,
            'total_corrected_edits': row.get('total_corrected_edits'),
            'comparisons': comparisons,
            'candidate_ready': all(item['beats'] for item in comparisons.values()) and false_edits is not None and false_edits <= 2,
        })
    ready = any(row['candidate_ready'] for row in policy_reports)
    return {
        'ready_for_external_sota_claims': ready,
        'message': 'Internal gate passed; still label ranked_sub_top_1 as first nonzero safe real-data correction, not SOTA competitive.' if ready else 'Not ready: a current HG002 policy must beat no_edit/consensus/support_rule on usable score with near-zero false edits.',
        'strict_no_recovery_is_baseline': True,
        'active_policy_label': 'ranked_sub_top_1 is the current best active HG002 policy only if it remains >0 corrected edits with 0 false edits.',
        'required_policy_files_are_distinct': len({row.get('source_file') for row in policy_reports if row.get('source_file')}) == len([row for row in policy_reports if row.get('source_file')]),
        'policy_reports': policy_reports,
    }

sota_readiness_report(HG002_AUDIT_PRESET)
